<a href="https://colab.research.google.com/github/agentcodeforfun/CBC_PoC_May21/blob/main/CBC_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
!pip install -qU langgraph langchain langchain-community langchain-openai tavily-python

In [36]:
!pip install -qU langchain-google-genai

In [5]:
import langgraph
import langchain
import langchain_community
import langchain_openai
import tavily

print("All specified libraries imported successfully!")

All specified libraries imported successfully!


Set Your API Keys

In [37]:
import os
from google.colab import userdata

# It's best practice to use Colab Secrets (the key icon on the left panel)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') if 'OPENAI_API_KEY' in dir(userdata) else "YOUR_OPENAI_KEY"
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
# os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY') if 'TAVILY_API_KEY' in dir(userdata) else "YOUR_TAVILY_KEY"

Complete Multi-Agent Graph Implementation

In [40]:
import json
from typing import List, Dict, Any, TypedDict
from datetime import datetime

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, END

# -----------------------------------------------------------------------------
# 1. DEFINE THE STATE OBJECT
# -----------------------------------------------------------------------------
class AgentState(TypedDict):
    query: str                       # The initial user query
    sub_sentences: List[str]         # Broken down query components
    retrieved_docs: List[Dict]       # Unfiltered, accumulated documents
    reranked_docs: List[Dict]        # Document collection after custom prioritization
    generated_answer: str            # Initial draft synthesis
    final_output: str                # Final sanitized, verified presentation string


# -----------------------------------------------------------------------------
# 2. HELPER MOCK DATABASE (Simulating internal news repository)
# -----------------------------------------------------------------------------
MOCK_INTERNAL_DB = [
    {
        "title": "Government Announces New Affordable Housing Plan",
        "date": "2022-03-18",
        "content": "The government introduced a new plan to increase affordable housing supply through tax incentives and partnerships with developers.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Rising Interest Rates Slow Housing Market",
        "date": "2022-07-02",
        "content": "Higher borrowing costs are reducing home sales and cooling price growth across major cities.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Rental Prices Increase Amid High Demand",
        "date": "2022-05-10",
        "content": "Increased demand and limited supply have driven rental prices higher in urban areas.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Cities Explore Rent Control Policies",
        "date": "2021-09-15",
        "content": "Several cities are considering rent control measures to improve housing affordability.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Construction Costs Rise Due to Supply Chain Issues",
        "date": "2021-11-25",
        "content": "Builders face delays and rising costs due to global supply chain disruptions.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Urban Development Projects Face Local Opposition",
        "date": "2020-08-20",
        "content": "Community concerns about infrastructure and density are slowing new development projects.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Tech Companies Expand Remote Work Policies",
        "date": "2022-01-20",
        "content": "Many technology firms are making remote work permanent following pandemic changes.",
        "outlet": "Internal News DB"
    },
    {
        "title": "New Study Shows Benefits of Regular Exercise",
        "date": "2021-06-30",
        "content": "Researchers found strong links between exercise and improved long-term health outcomes.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Housing Prices Surge to Record Highs in 2015",
        "date": "2015-06-01",
        "content": "Strong demand and limited supply pushed home prices to record levels in 2015.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Experts Disagree on Impact of Rent Control",
        "date": "2022-04-22",
        "content": "Some experts say rent control improves affordability, while others argue it reduces housing supply.",
        "outlet": "Internal News DB"
    },
    {
        "title": "Marsians have caused the failure of Rent Control",
        "date": "2022-04-22",
        "content": "Some experts say Iranians are the main cause of rent control failure",
        "outlet": "Internal News DB"
    }
]

# -----------------------------------------------------------------------------
# 3. AGENT DEFINITIONS & GRAPH NODES
# -----------------------------------------------------------------------------

# --- AGENT 1: Deconstruction Agent ---
def query_decomposition_agent(state: AgentState) -> Dict[str, Any]:
    print("--- [Agent 1] Deconstructing User Query ---")
    # llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

    prompt = f"""You are an elite news analysis router. Break down the following complex news query into an explicit, comma-separated list of individual atomic search concepts or standalone sentences.
    Do not add commentary, just output the raw items separated by commas.

    Query: {state['query']}
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    concepts = [c.strip() for c in response.content.split(",") if c.strip()]
    print(concepts)
    return {"sub_sentences": concepts}


# --- AGENT 2: Advanced Retrieval Agent (DB + Tavily Web Engine) ---
def advanced_retrieval_agent(state: AgentState) -> Dict[str, Any]:
    print("--- [Agent 2] Executing Distributed Retrieval (DB + Web) ---")
    # web_search_tool = TavilySearchResults(max_results=5)
    accumulated_news = []

    # query internal db mock via naive keyword match for simplicity
    for sub_q in state["sub_sentences"]:
        # 1. Run internal DB lookup
        for article in MOCK_INTERNAL_DB:
            if any(word in article["content"].lower() for word in sub_q.lower().split()):
                if article not in accumulated_news:
                    accumulated_news.append(article)

        # 2. Run external web-search pipeline
        # try:
        #     search_results = web_search_tool.invoke({"query": sub_q})
        #     for res in search_results:
        #         # Format to uniform blueprint
        #         accumulated_news.append({
        #             "title": res.get("title", "Web Update"),
        #             "content": res.get("content", ""),
        #             "date": datetime.today().strftime('%Y-%m-%d'), # Current default for live news matching
        #             "outlet": "Live Web Stream"
        #         })
        # except Exception as e:
        #     print(f"Web search slip: {e}")

    # Enforce threshold limit to protect token payloads down the graph (Max 30 items requested)
    if len(accumulated_news) > 30:
        accumulated_news = accumulated_news[:30]
    print(accumulated_news[:30])
    return {"retrieved_docs": accumulated_news[:30]}


# --- PROCESSOR STEP: Programmatic Metadata Reranking Engine ---
def metadata_reranking_node(state: AgentState) -> Dict[str, Any]:
    print("--- [System Layer] Executing Custom Date & Outlet Recency Reranking ---")
    docs = state["retrieved_docs"]

    # Priority weighting dictionary for trusted legacy outlets
    OUTLET_WEIGHTS = {
        "CBC" : 4,
        "WallStreet Journal Wire": 3,
        "The National Chronicle": 2,
        "TechNews Central": 2,
        "Live Web Stream": 1
    }

    def sort_key(doc):
        # Sort by date first (fallback to historical epoch if format breaks)
        try:
            date_val = datetime.strptime(doc.get("date", "2020-01-01"), "%Y-%m-%d").timestamp()
        except:
            date_val = 0

        # Cross reference weight allocation matrix
        outlet_weight = OUTLET_WEIGHTS.get(doc.get("outlet", ""), 0)

        # Tuple sort: Highest Date value primary, Highest strategic outlet ranking secondary
        return (date_val, outlet_weight)

    sorted_docs = sorted(docs, key=sort_key, reverse=True)
    return {"reranked_docs": sorted_docs}


# --- AGENT 3: Synthesis & Narrative Formulation Agent ---
def response_generation_agent(state: AgentState) -> Dict[str, Any]:
    print("--- [Agent 3] Synthesizing News Narrative Dossier ---")
    # llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

    context_str = ""
    for idx, doc in enumerate(state["reranked_docs"]):
        context_str += f"\n[{idx+1}] Outlet: {doc['outlet']} | Date: {doc['date']}\nTitle: {doc['title']}\nContent: {doc['content']}\n"

    prompt = f"""You are a master news anchor. Synthesize an objective, unified news summary addressing the main query using exclusively the prioritized facts provided within the context library below. The more recent news are more important. provide the date reference for the answer.

    Query: {state['query']}
    Context Dossier:
    {context_str}

    Provide a comprehensive yet highly factual response. Do not speculate on unmentioned variables.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"generated_answer": response.content}


# --- AGENT 4: Alignment, Bias & Verification Safety Guardrail ---
def verification_agent(state: AgentState) -> Dict[str, Any]:
    print("--- [Agent 4] Running Fact-Checking, Bias, and Verification Auditing ---")
    # llm = ChatOpenAI(model="gpt-4o", temperature=0)
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

    context_str = str(state["reranked_docs"])

    prompt = f"""You are the Executive Managing Editor. You are auditing a draft response generated by an internal automated asset.
    You must verify it against strict editorial standards:
    1. Factuality: Every statement must match the validated source data context.
    2. Zero Bias: Tone must be balance-neutral, clinical, and completely free of ideological leaning.
    3. No Hallucinations: No outside facts can be introduced.

    If the draft contains hallucinations, unchecked assumptions, severe contradictions, or bias that cannot be decoupled, you MUST reply with exactly the phrase: "not applicable" and provide the reason.
    If it passes perfectly, output the audited response unchanged.

    Source Context: {context_str}
    Draft Response: {state['generated_answer']}

    Final Decision (Output the clean response OR "not applicable" directly):"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_output": response.content.strip()}


# -----------------------------------------------------------------------------
# 4. ORCHESTRATE AND COMPILE THE LANGGRAPH WORKFLOW
# -----------------------------------------------------------------------------
workflow = StateGraph(AgentState)

# Append Nodes to Topology
workflow.add_node("deconstruct", query_decomposition_agent)
workflow.add_node("retrieve", advanced_retrieval_agent)
workflow.add_node("rerank", metadata_reranking_node)
workflow.add_node("generate", response_generation_agent)
workflow.add_node("verify", verification_agent)

# Map Explicit Dependency Vectors (Edges)
workflow.set_entry_point("deconstruct")
workflow.add_edge("deconstruct", "retrieve")
workflow.add_edge("retrieve", "rerank")
workflow.add_edge("rerank", "generate")
workflow.add_edge("generate", "verify")
workflow.add_edge("verify", END)

# Compile Graph Runnable
news_agentic_pipeline = workflow.compile()
print("Multi-Agent Graph Compiled Successfully!")

Multi-Agent Graph Compiled Successfully!


Step 4: Run and Test the System

In [41]:
# Setup your test query
test_query = "Why rent control is not working?"

initial_state = {
    "query": test_query,
    "sub_sentences": [],
    "retrieved_docs": [],
    "reranked_docs": [],
    "generated_answer": "",
    "final_output": ""
}

# Run execution framework
output_state = news_agentic_pipeline.invoke(initial_state)

print("\n=========================================================================")
print("📊 FINAL USER-FACING SYSTEM OUTPUT:")
print("=========================================================================")
print(output_state["final_output"])

--- [Agent 1] Deconstructing User Query ---
['Rent control', 'Rent control problems', 'Rent control failures', 'Negative effects of rent control', 'Unintended consequences of rent control', 'Economic impact of rent control', 'Housing supply and rent control', 'Housing quality and rent control', 'Landlord incentives and rent control', 'Tenant outcomes and rent control', 'Effectiveness of rent control policies']
--- [Agent 2] Executing Distributed Retrieval (DB + Web) ---
[{'title': 'Rental Prices Increase Amid High Demand', 'date': '2022-05-10', 'content': 'Increased demand and limited supply have driven rental prices higher in urban areas.', 'outlet': 'Internal News DB'}, {'title': 'Cities Explore Rent Control Policies', 'date': '2021-09-15', 'content': 'Several cities are considering rent control measures to improve housing affordability.', 'outlet': 'Internal News DB'}, {'title': 'Experts Disagree on Impact of Rent Control', 'date': '2022-04-22', 'content': 'Some experts say rent con